In [ ]:
# import sys
# sys.path.insert(0, "..")
# import logging

# from IPython import embed

# from opcua import Client


# if __name__ == "__main__":
#     logging.basicConfig(level=logging.WARN)
#     client = Client("opc.tcp://localhost:53530/OPCUA/SimulationServer/")
#     client.load_client_certificate("server_cert.pem")
#     client.load_private_key("mykey.pem")
#     try:
#         client.connect()
#         root = client.get_root_node()
#         objects = client.get_objects_node()
#         print("childs og objects are: ", objects.get_children())

#         embed()
#     finally:
#         client.disconnect()

In [6]:
from opcua import Client
from opcua.common.node import Node
from opcua import ua
from opcua.common.subscription import Subscription

    
client = Client("opc.tcp://192.168.115.80:4840")

client.secure_channel_timeout = 3*1000    # 一个请求的超时3s
client.session_timeout = 30*1000       # 30s超时

res = client.connect()
print(res)
root = client.get_root_node()
print(root)
objects_node = client.get_objects_node()
print(objects_node)
print(type(res))
objects_node.get_children()[2].get_children()[14].get_children()


Requested secure channel timeout to be 3000ms, got 300000ms instead


None
i=84
i=85
<class 'NoneType'>


[Node(FourByteNodeId(ns=3;i=5203)),
 Node(StringNodeId(ns=3;s="Buton_DB")),
 Node(StringNodeId(ns=3;s="A 上线位RFID_READER_FB_DB")),
 Node(StringNodeId(ns=3;s="B MT检验位RFID_READER_FB_DB_1")),
 Node(StringNodeId(ns=3;s="I 合格品下线位RFID_READER_FB_DB_2")),
 Node(StringNodeId(ns=3;s="J 不良品下线位RFID_READER_FB_DB_2")),
 Node(StringNodeId(ns=3;s="K UT+PT检验位RFID_READER_FB_DB_2")),
 Node(StringNodeId(ns=3;s="L 硬度检验位RFID_READER_FB_DB_2")),
 Node(StringNodeId(ns=3;s="M 尺寸外观检验位RFID_READER_FB_DB_2")),
 Node(StringNodeId(ns=3;s="MB_SERVER_DB")),
 Node(StringNodeId(ns=3;s="H MT检验位RFID_READER_FB_DB_2")),
 Node(StringNodeId(ns=3;s="C 合格品下线位RFID_READER_FB_DB_1")),
 Node(StringNodeId(ns=3;s="D 不良品下线位RFID_READER_FB_DB_1")),
 Node(StringNodeId(ns=3;s="E UT+PT检验位RFID_READER_FB_DB_1")),
 Node(StringNodeId(ns=3;s="F 硬度检验位RFID_READER_FB_DB_1")),
 Node(StringNodeId(ns=3;s="G 尺寸外观检验位RFID_READER_FB_DB_1"))]

In [4]:
# client.disconnect()

In [15]:
client.connect()
# servers = client.connect_and_find_servers()  # 不要在过程中使用

In [7]:
dbs = objects_node.get_children()[2].get_children()[13]
type(dbs)

opcua.common.node.Node

In [17]:
db502 = root.get_child(["0:Objects","3:KQL2-02PLC","3:DataBlocksGlobal","3:DB502_TCP_Server"])
path:Node = db502.get_path()[0]    # 获取当前path
print(path)
db502._make_relative_path(path)
# db502.get_parent().get_path()   # 获取父节点

ns=3;s="DB502_TCP_Server"


TypeError: 'Node' object is not iterable

Exception in thread Thread-6:
Traceback (most recent call last):
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\threading.py", line 980, in _bootstrap_inner
    self.run()
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\site-packages\opcua\client\client.py", line 66, in run
    self.client.open_secure_channel(renew=True)
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\site-packages\opcua\client\client.py", line 335, in open_secure_channel
    result = self.uaclient.open_secure_channel(params)
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\site-packages\opcua\client\ua_client.py", line 275, in open_secure_channel
    return self._uasocket.open_secure_channel(params)
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\site-packages\opcua\client\ua_client.py", line 207, in open_secure_channel
    self._send_request(request, message_type=ua.MessageType.SecureOpen, callback=clb)
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\site-packages\opcua\clien

In [8]:
string_struct:Node = db502.get_variables()[-2]

In [4]:
shangxianwei:Node = string_struct.get_variables()[0]

In [5]:
shangxianwei.get_value()

'0000000000'

In [11]:
heart_beat:Node = db502.get_variables()[0]
# 读取变量
heart_beat.get_value()

ConnectionResetError: [WinError 10054] 远程主机强迫关闭了一个现有的连接。

In [12]:
# 写入变量
value = ua.DataValue(ua.Variant(200,ua.VariantType.UInt16))
# heart_beat.set_value(value=100,varianttype=ua.VariantType.UInt16)     # 不能用
heart_beat.set_value(value)

Python: New data change event ns=3;s="DB502_TCP_Server"."Temp1" 200


In [28]:
# 字符串读写
string_node :Node= db502.get_variables()[-2].get_variables()[10]
print(string_node.get_value())
value = ua.DataValue(ua.Variant('1234567890',ua.VariantType.String))
string_node.set_value(value)
print(string_node.get_value())

1234567890
1234567890


In [25]:
import time
class SubHandler(object):

    """
    Subscription Handler. To receive events from server for a subscription
    data_change and event methods are called directly from receiving thread.
    Do not do expensive, slow or network operation there. Create another 
    thread if you need to do such a thing
    """

    def datachange_notification(self, node:Node, val, data):
        print("Python: New data change event", node, val)
        if val==100:
            print(1111111111)
        # 然后再复位该变量
        

    def event_notification(self, event):
        print("Python: New event", event)


# subscribing to a variable node
handler = SubHandler()
sub = client.create_subscription(500, handler)    # 500ms的订阅
handle = sub.subscribe_data_change(heart_beat)    # 订阅
# sub.unsubscribe(handle=handle)    # 取消订阅
print(type(handle))
time.sleep(0.1)

# we can also subscribe to events from server
# sub.subscribe_events()

# sub.unsubscribe(handle)
# sub.delete()


<class 'int'>


Python: New data change event ns=3;s="DB502_TCP_Server"."Temp1" 200
Python: New data change event ns=3;s="DB502_TCP_Server"."Temp1" 256
Python: New data change event ns=3;s="DB502_TCP_Server"."Temp1" 100
1111111111


In [14]:
handle

1

In [30]:
heart_beat.get_value()

Exception in thread Thread-6:
Traceback (most recent call last):
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\threading.py", line 980, in _bootstrap_inner
    self.run()
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\site-packages\opcua\client\client.py", line 66, in run
    self.client.open_secure_channel(renew=True)
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\site-packages\opcua\client\client.py", line 335, in open_secure_channel
    result = self.uaclient.open_secure_channel(params)
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\site-packages\opcua\client\ua_client.py", line 275, in open_secure_channel
    return self._uasocket.open_secure_channel(params)
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\site-packages\opcua\client\ua_client.py", line 207, in open_secure_channel
    self._send_request(request, message_type=ua.MessageType.SecureOpen, callback=clb)
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\site-packages\opcua\clien

TimeoutError: 

In [ ]:
heart_beat.nodeid

In [19]:
client.disconnect()

ServiceFault from server received while waiting for publish response
exception calling callback for <Future at 0x178d91b29a0 state=finished returned Buffer>
Traceback (most recent call last):
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\concurrent\futures\_base.py", line 330, in _invoke_callbacks
    callback(self)
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\site-packages\opcua\client\ua_client.py", line 493, in _call_publish_callback
    self._uasocket.check_answer(data, "while waiting for publish response")
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\site-packages\opcua\client\ua_client.py", line 93, in check_answer
    hdr.ServiceResult.check()
  File "c:\Users\weili\miniconda3\envs\zhijianxian\lib\site-packages\opcua\ua\uatypes.py", line 218, in check
    raise UaStatusCodeError(self.value)
opcua.ua.uaerrors._auto.BadSessionClosed: "The session was closed by the client."(BadSessionClosed)
ServiceFault from server received while waiting for publish re